In [ ]:
import cv2
import win32com.client
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
from ultralytics import YOLO
import time
import threading
import keyboard

# 1. Setup Audio (Windows built-in)
speaker = win32com.client.Dispatch("SAPI.SpVoice")

def speak(text):
    def run():
        speaker.Speak(text)
    t = threading.Thread(target=run)
    t.start()

# 2. Load model
model_path = r'.\ml_pipeline_Obstacle_det\runs\detect\Drishti_Final_Push\v11n_augmented_852\weights\best.pt'
model = YOLO(model_path)

# 3. Open Webcam
cap = cv2.VideoCapture(0)
last_announced = {}

print("Camera started! Press Q anytime to stop.")

try:
    while cap.isOpened():

        # Press Q to stop
        if keyboard.is_pressed('q'):
            print("Q pressed - Stopping camera...")
            break

        ret, frame = cap.read()
        if not ret:
            break

        # 4. Predict
        results = model.predict(frame, conf=0.5, verbose=False)

        # 5. Announce & print
        detected_classes = []
        for box in results[0].boxes:
            class_id = int(box.cls[0])
            class_name = model.names[class_id]
            detected_classes.append(class_name)

        for class_name in set(detected_classes):
            now = time.time()
            if class_name not in last_announced or (now - last_announced[class_name]) > 3:
                # Customize your message here!
                speak(f"Hey there! Its a {class_name}")
                print(f"Successfully detected: {class_name}")
                last_announced[class_name] = now

        # 6. Display via matplotlib
        annotated_frame = results[0].plot()
        frame_rgb = cv2.cvtColor(annotated_frame, cv2.COLOR_BGR2RGB)

        plt.figure(figsize=(8, 5))
        plt.imshow(frame_rgb)
        plt.axis('off')
        display(plt.gcf())
        clear_output(wait=True)
        plt.close()

finally:
    cap.release()
    print("Camera stopped successfully.")

Q pressed - Stopping camera...
Camera stopped successfully.


In [25]:
%pip install keyboard

Note: you may need to restart the kernel to use updated packages.
